# Neural-DTB oscillatory potential-game experiment

This notebook runs the controlled two-player experiment on Google Colab. It compares the standard high-accuracy RK4 solution with particles pushed forward by the existing Neural Deep Tangent Bundle (DTB) method.

The default `SMOKE = True` checks all eight `(gamma, omega)` cases quickly. After that succeeds, set `SMOKE = False` to run the full `N=2000`, `T=1` experiment.

## 1. Clone the game-dynamics branch and install dependencies

In [ ]:
import pathlib
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/sun-mengwei/dtb-colab-experiments.git"
BRANCH = "codex/game-dynamics-dtb"
REPO_DIR = pathlib.Path("/content/dtb-colab-experiments")

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO_DIR)],
    check=True,
)
%cd /content/dtb-colab-experiments/dtb_game_dynamics_unnormalized
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt", "pytest", "pandas"],
    check=True,
)
print("Repository and dependencies are ready.")

## 2. Verify the implementation

These tests include the analytic potential-gradient check, analytic game-Jacobian check, reproducible initial particles, controlled neural initialization, and an end-to-end DTB runner check.

In [ ]:
subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)

## 3. Game definition

For $x=(x_1,x_2)\in[-1,1]^2$, both players use the potential

$$\Phi_{\omega,\gamma}(x)=-\frac{\lambda}{2}(x_1^2+x_2^2)-\frac{\gamma}{2}(x_1-x_2)^2+\frac{\varepsilon}{\omega}[\cos(\omega x_1)+\cos(\omega x_2)].$$

The deterministic game velocity is $b=\nabla\Phi$:

$$b_1=-\lambda x_1-\gamma(x_1-x_2)-\varepsilon\sin(\omega x_1),$$

$$b_2=-\lambda x_2-\gamma(x_2-x_1)-\varepsilon\sin(\omega x_2).$$

The DTB diffusion matrix is zero for this example, so its target velocity is exactly this game velocity.

## 4. Controlled experiment settings

Only `OMEGA_MULTIPLIERS` and `GAMMAS` change across the eight cases. The runner verifies that every case starts from the same particles and the same neural-network initialization.

In [ ]:
# Start with the smoke run. Set False only after it succeeds.
SMOKE = True
RESET_OUTPUT = True

# Game sweep
OMEGA_MULTIPLIERS = [2, 4, 8, 16]  # omega = multiplier*pi
GAMMAS = [0.0, 0.2]
LAMBDA_VALUE = 0.5
EPSILON = 0.5

# Shared particle/time configuration (used when SMOKE is False)
PARTICLES = 2000
STEPS = 200
STEP_SIZE = 0.005  # final time T=1
SEED = 2026
MODEL_SEED = 91

# Existing neural tangent representation
ARCHITECTURE = "mlp"
WIDTH = 32
DEPTH = 4
BASIS_SIZE = 128
SVD_RTOL = 1e-3
ACTIVATION = "tanh"

# Zero keeps the network fixed; positive values enable existing periodic refits.
REFIT_INTERVAL = 0
REFERENCE_SUBSTEPS = 20
DEVICE = "auto"
DTYPE = "float32"

OUTPUT_ROOT = pathlib.Path(
    "outputs/oscillatory_potential_game_smoke"
    if SMOKE
    else "outputs/oscillatory_potential_game_full"
)
print("Outputs will be written to", OUTPUT_ROOT)

## 5. Run DTB and the standard ODE reference

For every initial particle, the reference trajectory uses vectorized classical RK4 with a step size `STEP_SIZE / REFERENCE_SUBSTEPS`. A second reference with twice as many substeps checks convergence. The DTB settings remain unchanged.

In [ ]:
if RESET_OUTPUT and OUTPUT_ROOT.exists():
    shutil.rmtree(OUTPUT_ROOT)

command = [
    sys.executable,
    "run_oscillatory_potential_game.py",
    "--omega-multipliers", *map(str, OMEGA_MULTIPLIERS),
    "--gammas", *map(str, GAMMAS),
    "--lambda-value", str(LAMBDA_VALUE),
    "--epsilon", str(EPSILON),
    "--particles", str(PARTICLES),
    "--steps", str(STEPS),
    "--step-size", str(STEP_SIZE),
    "--seed", str(SEED),
    "--model-seed", str(MODEL_SEED),
    "--architecture", ARCHITECTURE,
    "--width", str(WIDTH),
    "--depth", str(DEPTH),
    "--basis-size", str(BASIS_SIZE),
    "--svd-rtol", str(SVD_RTOL),
    "--activation", ACTIVATION,
    "--refit-interval", str(REFIT_INTERVAL),
    "--reference-substeps", str(REFERENCE_SUBSTEPS),
    "--device", DEVICE,
    "--dtype", DTYPE,
    "--output-root", str(OUTPUT_ROOT),
]
if SMOKE:
    command.append("--smoke")

print("Running:", " ".join(command))
subprocess.run(command, check=True)

## 6. Summary and cross-frequency diagnostics

Do not draw scientific conclusions from the smoke run. The full run reports projection residual, trajectory error, basin-mass error, numerical rank, retained condition number, coefficient norm, potential checks, reference self-error, and runtime.

In [ ]:
import pandas as pd
from IPython.display import Image, Markdown, display

summary = pd.read_csv(OUTPUT_ROOT / "tables" / "summary.csv")
display(summary)
display(Markdown((OUTPUT_ROOT / "report.md").read_text()))

for filename in [
    "gamma_0_frequency_comparison.png",
    "gamma_0p2_frequency_comparison.png",
    "final_error_vs_frequency.png",
]:
    path = OUTPUT_ROOT / "figures" / filename
    print(filename)
    display(Image(filename=str(path)))

## 7. Compare reference and DTB particle snapshots

Choose any `(gamma, omega)` case below. In `particle_comparison.png`, the **top row is the standard RK4 solution** and the **bottom row is the DTB particle pushforward**. The columns show `t=0`, `t=T/2`, and `t=T` using identical axis limits and initial particles.

In [ ]:
INSPECT_GAMMA = 0.2
INSPECT_OMEGA_MULTIPLIER = 16

def safe_number(value):
    return f"{value:g}".replace("-", "m").replace(".", "p")

case_name = (
    pathlib.Path(f"gamma_{safe_number(INSPECT_GAMMA)}")
    / f"omega_{safe_number(INSPECT_OMEGA_MULTIPLIER)}pi"
)
case_figures = OUTPUT_ROOT / "figures" / "cases" / case_name
case_results = OUTPUT_ROOT / "results" / case_name

display(Image(filename=str(case_figures / "particle_comparison.png")))

## 8. Inspect the potential, vector field, and numerical diagnostics

In [ ]:
display(Image(filename=str(case_figures / "potential_vector_field.png")))
display(Image(filename=str(case_figures / "diagnostics.png")))
display(pd.read_csv(case_results / "equilibria.csv").head(30))

## 9. Inspect raw trajectories (optional)

The `.npz` history contains both complete particle paths, trajectory/projection errors, potential histories, SVD statistics, and basin diagnostics.

In [ ]:
import numpy as np

with np.load(case_results / "history.npz") as history:
    print("DTB particles:", history["dtb_particles"].shape)
    print("RK4 particles:", history["reference_particles"].shape)
    print("Times:", history["times"].shape)
    print("Final trajectory RMSE:", history["trajectory_rmse"][-1])
    print("Mean projection residual:", history["projection_residuals"].mean())

## 10. Save or download outputs (optional)

In [ ]:
SAVE_TO_DRIVE = False
DOWNLOAD_ZIP = False

if SAVE_TO_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    destination = pathlib.Path("/content/drive/MyDrive") / OUTPUT_ROOT.name
    if destination.exists():
        shutil.rmtree(destination)
    shutil.copytree(OUTPUT_ROOT, destination)
    print("Saved to", destination)

if DOWNLOAD_ZIP:
    from google.colab import files

    archive = shutil.make_archive(str(OUTPUT_ROOT), "zip", root_dir=OUTPUT_ROOT)
    files.download(archive)